# Hydraulic Surrogate Model

This example demonstrates how to load a pre-trained hydraulic surrogate model
and use it for predicting flows and pressure heads.

In [ ]:
%pip install epyt-control[hydsurrogate] --quiet

In [ ]:
from epyt_control.models import PIGNNModel
from epyt_flow.utils import plot_timeseries_data
import numpy as np

from anytown_helper import create_test_data_anytown

Create and load a pre-trained hydrauic surrogate model for Anytown by creating a new instance of [epyt_control.models.PIGNNModel](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.pi_gnn_hydsurrogate.PIGNNModel):

In [ ]:
model = PIGNNModel.from_network("anytown", load_pretrained_model=True)
#model.load_model("anytown_pignn.pt")  # Alternatively, you can load your own pre-trained weights

Create test data:

In [ ]:
heads, reservoir_idx, demands, flows = create_test_data_anytown()  # First axis in 'heads', 'demands', and 'flows' encode time!

Use the hydraulic surrogate model to predict flow rates (at every link), heads and demands at every node -- we use the [.predict_as_numpy](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.pi_gnn_hydsurrogate.PIGNNModel.predict) function:

In [ ]:
heads_pred, demands_pred, flows_pred = model.predict_as_numpy(reservoir_heads=heads[:, reservoir_idx],
                                                              demands=demands)

Compare predictions to ground truth:

In [ ]:
plot_timeseries_data(np.abs(heads_pred - heads),
                     x_axis_label="Time",
                     y_axis_label="Absolute pressure head error")

plot_timeseries_data(np.abs(flows_pred - flows),
                     x_axis_label="Time",
                     y_axis_label="Absolute flow rate error")

Alternatively, we can also use the [.predict_as_scada_data()](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.pi_gnn_hydsurrogate.PIGNNModel.predict_as_scada_data) function to get the results as a [epyt_flow.simulation.ScadaData](https://epyt-flow.readthedocs.io/en/stable/epyt_flow.simulation.scada.html#epyt_flow.simulation.scada.scada_data.ScadaData) object:

In [ ]:
scada_data = model.predict_as_scada_data(reservoir_heads=heads[:, reservoir_idx],
                                         demands=demands)

# Plot pressure and flow
scada_data.plot_pressures()
scada_data.plot_flows()